In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [2]:
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
dataset = pd.get_dummies(dataset,drop_first = True)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,True
1,15810944,35,20000,0,True
2,15668575,26,43000,0,False
3,15603246,27,57000,0,False
4,15804002,19,76000,0,True
...,...,...,...,...,...
395,15691863,46,41000,1,False
396,15706071,51,23000,1,True
397,15654296,50,20000,1,False
398,15755018,36,33000,0,True


In [4]:
dataset = dataset.drop("User ID", axis = 1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,True
1,35,20000,0,True
2,26,43000,0,False
3,27,57000,0,False
4,19,76000,0,True
...,...,...,...,...
395,46,41000,1,False
396,51,23000,1,True
397,50,20000,1,False
398,36,33000,0,True


In [5]:
dataset['Purchased'].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [6]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [7]:
independent = dataset[['Age', 'EstimatedSalary','Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent.shape

(400, 3)

In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split (independent, dependent, test_size =1/3, random_state =0 )

In [11]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [12]:
from sklearn.linear_model import LogisticRegression

In [13]:
from sklearn.model_selection import GridSearchCV
param_grid = {'solver':['liblinear', 'newton-cg', 'saga', 'lbfgs'],
                          'penalty':['l2'], 'C': [0.1, 1, 10]}
grid = GridSearchCV(LogisticRegression(), param_grid, verbose = 3, refit = True, n_jobs = -1, scoring = 'accuracy')

grid.fit(x_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,LogisticRegression()
,param_grid,"{'C': [0.1, 1, ...], 'penalty': ['l2'], 'solver': ['liblinear', 'newton-cg', ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [17]:
re = grid.cv_results_
grid_predictions = grid.predict(x_test)


from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

print(cm)
print(clf_report)

[[79  6]
 [10 39]]
              precision    recall  f1-score   support

           0       0.89      0.93      0.91        85
           1       0.87      0.80      0.83        49

    accuracy                           0.88       134
   macro avg       0.88      0.86      0.87       134
weighted avg       0.88      0.88      0.88       134



In [19]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test, grid_predictions, average = 'weighted')
print("the f1_macro value for best parameter {}: " . format(grid.best_params_), f1_macro)  

the f1_macro value for best parameter {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}:  0.8794289739855382


In [23]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])

0.9507803121248499

In [24]:
table = pd.DataFrame.from_dict(re)
table


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.007422,0.001112,0.005346,0.001485,0.1,l2,liblinear,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline...",0.814815,0.792453,0.679245,0.924528,0.924528,0.827114,0.091866,9
1,0.015234,0.002942,0.006012,0.000949,0.1,l2,newton-cg,"{'C': 0.1, 'penalty': 'l2', 'solver': 'newton-...",0.777778,0.754717,0.716981,0.924528,0.867925,0.808386,0.076428,10
2,0.009378,0.002126,0.005263,0.000481,0.1,l2,saga,"{'C': 0.1, 'penalty': 'l2', 'solver': 'saga'}",0.777778,0.754717,0.716981,0.924528,0.867925,0.808386,0.076428,10
3,0.012413,0.003723,0.005874,0.000696,0.1,l2,lbfgs,"{'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.777778,0.754717,0.716981,0.924528,0.867925,0.808386,0.076428,10
4,0.005743,0.000596,0.006350,0.001268,1.0,l2,liblinear,"{'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}",0.814815,0.811321,0.698113,0.924528,0.924528,0.834661,0.084541,1
5,0.015571,0.000669,0.006110,0.001916,1.0,l2,newton-cg,"{'C': 1, 'penalty': 'l2', 'solver': 'newton-cg'}",0.814815,0.811321,0.698113,0.924528,0.924528,0.834661,0.084541,1
6,0.008722,0.000870,0.006941,0.001180,1.0,l2,saga,"{'C': 1, 'penalty': 'l2', 'solver': 'saga'}",0.814815,0.811321,0.698113,0.924528,0.924528,0.834661,0.084541,1
7,0.014208,0.003815,0.006512,0.000933,1.0,l2,lbfgs,"{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}",0.814815,0.811321,0.698113,0.924528,0.924528,0.834661,0.084541,1
8,0.007129,0.001164,0.005957,0.001704,10.0,l2,liblinear,"{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}",0.814815,0.811321,0.698113,0.924528,0.924528,0.834661,0.084541,1
9,0.015949,0.001871,0.005996,0.001726,10.0,l2,newton-cg,"{'C': 10, 'penalty': 'l2', 'solver': 'newton-cg'}",0.814815,0.811321,0.698113,0.924528,0.924528,0.834661,0.084541,1


In [25]:
age_input = float(input("Age:"))
salary_input= float (input("EstimatedSalary:"))
sex_male_input = int(input("Gender_Male:"))

Age: 22
EstimatedSalary: 32200
Gender_Male: 1


In [26]:
future_prediction = grid.predict([[ age_input, salary_input, sex_male_input]])

In [27]:
future_prediction

array([1])